# 📄 FILE: 03_train.py

## 🎯 Chức năng
Đây là "phòng tập gym" cho mô hình AI. Nó sẽ lấy dữ liệu và ép mô hình học đi học lại (Epochs) cho đến khi đạt kết quả tốt nhất.

## ⚙️ Quy trình hoạt động
1.  **Import Dynamic:** Sử dụng `importlib` để lấy Class `Dataset` và `Model` từ file `02_model_architecture.py` (Xử lý tên file có số).
2.  **Split Data:** Chia dữ liệu thành 80% để học (Train) và 20% để kiểm tra (Validation).
3.  **Optimizer:** Sử dụng `AdamW` (thuật toán tối ưu phổ biến cho BERT).
4.  **Training Loop:**
    * Mô hình đoán -> Tính sai số (Loss) -> Sửa lỗi (Backward).
    * Sau mỗi vòng, kiểm tra trên tập Validation.
5.  **Save Best Model:** Chỉ lưu lại phiên bản mô hình có độ chính xác cao nhất vào file `best_model.pth`.

In [1]:
# ==============================================================================
# FILE: 03_train.py
# CHỨC NĂNG:
# 1. Load dữ liệu chuẩn.
# 2. Huấn luyện mô hình (Training).
# 3. Lưu model tốt nhất (best_model.pth).
# ==============================================================================

In [2]:
import pandas as pd
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from torch.optim import AdamW
import importlib

In [3]:
try:
    arch = importlib.import_module("02_model_architecture")
    DiabetesDataset = arch.DiabetesDataset
    DiabetesHybridModel = arch.DiabetesHybridModel
    MODEL_NAME = arch.MODEL_NAME
    MAX_LEN = arch.MAX_LEN
    print("✅ Đã import thành công các Class từ file 02.")
except ImportError:
    print("❌ Lỗi: Không tìm thấy file '02_model_architecture.py'. Hãy kiểm tra lại tên file.")
    exit()

✅ Đã import thành công các Class từ file 02.


In [4]:
# --- CẤU HÌNH HUẤN LUYỆN (Hyperparameters) ---
BATCH_SIZE = 4        # Số lượng câu học trong 1 lần (Batch)
EPOCHS = 10             # Số lần học lại toàn bộ dữ liệu (Dữ liệu ít nên tăng lên 10)
LEARNING_RATE = 2e-5   # Tốc độ học (Thấp để tránh học vẹt)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Thiết bị sử dụng: {DEVICE}")

Thiết bị sử dụng: cuda


In [5]:
# --- DỌN DẸP BỘ NHỚ TRƯỚC KHI CHẠY ---
import gc
torch.cuda.empty_cache()
gc.collect()
print(f"Đã dọn dẹp VRAM. Thiết bị sử dụng: {DEVICE}")

Đã dọn dẹp VRAM. Thiết bị sử dụng: cuda


In [6]:
# --- BƯỚC 1: CHUẨN BỊ DỮ LIỆU ---
def create_data_loader(df, tokenizer, batch_size):
    ds = DiabetesDataset(
        texts=df.text.to_numpy(),
        labels=df.outcome.to_numpy(),
        stages=df.stage.to_numpy(),
        keyword_weights_path='diabetes_keywords.json', # File từ điển tạo ở file 01
        tokenizer=tokenizer
    )
    return DataLoader(ds, batch_size=batch_size, num_workers=0) # num_workers=0 để chạy ổn định trên Windows

# Load dữ liệu
try:
    df = pd.read_csv('../datasets/raw/symptoms2.csv')
    print(f"Đã load {len(df)} dòng dữ liệu.")
except:
    print("Chưa có file 'symptoms2.csv'. Hãy chạy file 01 trước!")
    exit()

# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Chia tập Train (90%) - Test (10%)
# Stratify theo 'outcome' để đảm bảo tỉ lệ bệnh đều nhau ở 2 tập
train_df, val_df = train_test_split(df, test_size=0.1, random_state=42, stratify=df['outcome'])

train_loader = create_data_loader(train_df, tokenizer, BATCH_SIZE)
val_loader = create_data_loader(val_df, tokenizer, BATCH_SIZE)

Đã load 367 dòng dữ liệu.


In [ ]:
# --- BƯỚC 2: KHỞI TẠO MÔ HÌNH ---
model = DiabetesHybridModel(n_classes=2) # Dự đoán 2 lớp: Bệnh / Không bệnh
model = model.to(DEVICE)

# Cấu hình tối ưu hóa (Optimizer)
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=0, num_training_steps=total_steps
)
loss_fn = nn.CrossEntropyLoss().to(DEVICE) # Hàm mất mát

config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


pytorch_model.bin:   0%|          | 0.00/581M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Exception ignored in: <function tqdm.__del__ at 0x0000025C2993DBD0>
Traceback (most recent call last):
  File "d:\IT\HK1_Y4\Class\Project_1\Project_2\.venv\lib\site-packages\tqdm\std.py", line 1148, in __del__
    self.close()
  File "d:\IT\HK1_Y4\Class\Project_1\Project_2\.venv\lib\site-packages\tqdm\notebook.py", line 279, in close
    self.disp(bar_style='danger', check_delay=False)
AttributeError: 'tqdm' object has no attribute 'disp'


In [9]:
def train_epoch(model, data_loader, n_examples):
    model = model.train()
    losses = []
    correct_predictions = 0
    
    for d in data_loader:
        input_ids = d["input_ids"].to(DEVICE)
        attention_mask = d["attention_mask"].to(DEVICE)
        dict_features = d["dict_features"].to(DEVICE)
        targets = d["labels"].to(DEVICE)

        # Forward pass (Mô hình dự đoán)
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            dict_features=dict_features
        )
        
        _, preds = torch.max(outputs, dim=1)
        loss = loss_fn(outputs, targets)

        correct_predictions += torch.sum(preds == targets)
        losses.append(loss.item())

        # Backward pass (Mô hình học rút kinh nghiệm)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

    return correct_predictions.double() / n_examples, np.mean(losses)

def eval_model(model, data_loader, n_examples):
    model = model.eval()
    losses = []
    correct_predictions = 0

    with torch.no_grad():
        for d in data_loader:
            input_ids = d["input_ids"].to(DEVICE)
            attention_mask = d["attention_mask"].to(DEVICE)
            dict_features = d["dict_features"].to(DEVICE)
            targets = d["labels"].to(DEVICE)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                dict_features=dict_features
            )
            _, preds = torch.max(outputs, dim=1)
            loss = loss_fn(outputs, targets)
            correct_predictions += torch.sum(preds == targets)
            losses.append(loss.item())

    return correct_predictions.double() / n_examples, np.mean(losses)

In [10]:
# --- BƯỚC 4: CHẠY TRAIN & LƯU MODEL ---
history = {'train_acc': [], 'train_loss': [], 'val_acc': [], 'val_loss': []}
best_accuracy = 0

print("\n🚀 BẮT ĐẦU HUẤN LUYỆN...")
print("-" * 30)

for epoch in range(EPOCHS):
    print(f'Epoch {epoch + 1}/{EPOCHS}')
    
    train_acc, train_loss = train_epoch(model, train_loader, len(train_df))
    print(f'Train loss {train_loss:.4f} | Accuracy {train_acc:.4f}')

    val_acc, val_loss = eval_model(model, val_loader, len(val_df))
    print(f'Val   loss {val_loss:.4f} | Accuracy {val_acc:.4f}')

    # Lưu lại model nếu kết quả tốt nhất từ trước tới giờ
    if val_acc > best_accuracy:
        torch.save(model.state_dict(), 'NLP_model.pth')
        best_accuracy = val_acc
        print("💾 Đã lưu model tốt nhất!")
    
    print("-" * 30)

print(f"\n🏆 Huấn luyện hoàn tất! Độ chính xác cao nhất: {best_accuracy:.4f}")
print("👉 Hãy chuyển sang File 04 để chạy thử nghiệm.")


🚀 BẮT ĐẦU HUẤN LUYỆN...
------------------------------
Epoch 1/10
Train loss 0.6529 | Accuracy 0.6091
Val   loss 0.5842 | Accuracy 0.6757
💾 Đã lưu model tốt nhất!
------------------------------
Epoch 2/10
Train loss 0.3932 | Accuracy 0.8424
Val   loss 0.4991 | Accuracy 0.8108
💾 Đã lưu model tốt nhất!
------------------------------
Epoch 3/10
Train loss 0.2099 | Accuracy 0.9424
Val   loss 0.3631 | Accuracy 0.9189
💾 Đã lưu model tốt nhất!
------------------------------
Epoch 4/10
Train loss 0.1470 | Accuracy 0.9697
Val   loss 0.3128 | Accuracy 0.9189
------------------------------
Epoch 5/10
Train loss 0.0644 | Accuracy 0.9879
Val   loss 0.3295 | Accuracy 0.9189
------------------------------
Epoch 6/10
Train loss 0.0228 | Accuracy 0.9970
Val   loss 0.4395 | Accuracy 0.8919
------------------------------
Epoch 7/10
Train loss 0.0205 | Accuracy 0.9970
Val   loss 0.3516 | Accuracy 0.9189
------------------------------
Epoch 8/10
Train loss 0.0198 | Accuracy 0.9970
Val   loss 0.3501 | Accu